# Problem Framing (Business Context)

##### Fictitious business case:
A subscription-based e-commerce platform wants to predict customer churn.

##### Target variable:

- churn (1 = customer churned, 0 = retained)

##### Typical business questions supported by features:

- Who is likely to churn?

- Which customer behaviors correlate with churn?

- How do pricing, tenure, and engagement interact?

# Generate a Fictitious Dataset

Features include:

- Missing values

- Skewed distributions

- Mixed feature types

- Business-realistic variable

In [17]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "age": np.random.randint(18, 75, size=n),
    "monthly_income": np.random.lognormal(mean=8, sigma=0.5, size=n),
    "tenure_months": np.random.randint(1, 120, size=n),
    "avg_monthly_spend": np.random.gamma(shape=2, scale=50, size=n),
    "num_support_tickets": np.random.poisson(lam=2, size=n),
    "contract_type": np.random.choice(
        ["month-to-month", "annual", "biennial"], size=n, p=[0.6, 0.3, 0.1]
    ),
    "payment_method": np.random.choice(
        ["credit_card", "debit_card", "paypal", "bank_transfer"], size=n
    ),
    "region": np.random.choice(
        ["North", "South", "East", "West"], size=n
    ),
    "has_discount": np.random.choice([0, 1], size=n, p=[0.7, 0.3])
})

# Target variable with business logic
data["churn"] = (
    (data["tenure_months"] < 12).astype(int)
    | (data["num_support_tickets"] > 4).astype(int)
).astype(int)

# Inject missing values
for col in ["age", "monthly_income", "avg_monthly_spend"]:
    data.loc[data.sample(frac=0.05).index, col] = np.nan

data.head()


,customer_id,age,monthly_income,tenure_months,avg_monthly_spend,num_support_tickets,contract_type,payment_method,region,has_discount,churn
0,1,56.0,1060.270083,75,162.418321,2,annual,paypal,North,1,0
1,2,NaN,2851.042456,86,237.169358,3,month-to-month,debit_card,West,0,0
2,3,46.0,1552.722700,22,81.640444,2,annual,bank_transfer,North,0,0
3,4,NaN,4166.519335,107,102.757373,3,annual,bank_transfer,South,0,0
4,5,60.0,3580.648195,89,56.336590,2,month-to-month,debit_card,North,1,0


In [18]:
data.shape

(1000, 11)

# Feature Categorization (Critical Step)

In [19]:
target = "churn"
id_cols = ["customer_id"]

numerical_features = [
    "age",
    "monthly_income",
    "tenure_months",
    "avg_monthly_spend",
    "num_support_tickets"
]

categorical_features = [
    "contract_type",
    "payment_method",
    "region"
]

binary_features = ["has_discount"]


# Business-Driven Feature Engineering
## Ratio & Interaction Features

In [20]:
data["spend_per_month_of_tenure"] = (
    data["avg_monthly_spend"] / (data["tenure_months"] + 1)
)

data["tickets_per_month"] = (
    data["num_support_tickets"] / (data["tenure_months"] + 1)
)


__Business rationale:__

- Normalize behavior over time

- Identify “high friction” customers early


## Customer Value Segmentation

In [21]:
data["customer_value_segment"] = pd.qcut(
    data["monthly_income"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"]
)


__Business rationale:__
- Aligns with marketing segmentation and pricing strategies.


## Behavioral Flags

In [22]:
data["high_support_user"] = (data["num_support_tickets"] >= 5).astype(int)
data["short_tenure"] = (data["tenure_months"] < 12).astype(int)


# . Preprocessing Strategy (ML-Oriented)
Key Design Principles


- No data leakage


- Pipeline-based


- Reusable across models


- Scalable

# Sklearn Pipelines & Transformers

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from sklearn.impute import SimpleImputer


## Numerical Pipeline

In [24]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

__Reason:__
- Median is robust to outliers

- Scaling benefits linear models, SVMs, neural networks

## Categorical Pipelines
__Nominal Categorical (No Order)__

In [25]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


__Ordinal / Business-Defined Order__

In [26]:
value_segment_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=[["low", "mid_low", "mid_high", "high"]]
    ))
])


## Binary Features

In [27]:
binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])


# ColumnTransformer Assembly

In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features + [
            "spend_per_month_of_tenure",
            "tickets_per_month"
        ]),
        ("cat", categorical_pipeline, categorical_features),
        ("ord", value_segment_pipeline, ["customer_value_segment"]),
        ("bin", binary_pipeline, binary_features + [
            "high_support_user",
            "short_tenure"
        ])
    ],
    remainder="drop"
)


 # Full Logistic Regression ML Pipeline 

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

X = data.drop(columns=[target] + id_cols)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'monthly_income',
                                                   'tenure_months',
                                                   'avg_monthly_spend',
                                                   'num_support_tickets',
                                                   'spend_per_month_of_tenure',
                                                   'tickets_per_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strateg...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(categories=[['low',
                                                                                               'mid_low',
                                                                                               'mid_high',
                                                                                               'high']]))]),
                                                  ['customer_value_segment']),
                                                 ('bin',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['has_discount',
                                                   'high_support_user',
                                                   'short_tenure'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [35]:
y_pred = model.predict(X_test)

accuracy_score(y_test, y_pred)

0.985